In [ ]:
# 'adult' | 'bank_marketing' | 'california_housing' | 'cardiotocography' | 'compas' | 'loan_prediction'
DATASET = 'bank_marketing'

LLM_FRAMEWORKS =  ['great', 'taptap', 'tabula']
LLMS =['distilgpt2', 'taptap_distill', 'qwen3_03B_distil']
GANS = ['ctgan', 'tvae', 'tabfairgan', 'ctabgan', 'ctabgan_plus']


In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "datasets"))

import synth_eval as se

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ── Per-dataset configs (only what is specific to each dataset) ─────
CONFIGS = {
    'adult': {
        'dataset_name': 'adult',
        'seeds':        [42, 43, 44],
        'target':       'income',
        'task':         'classification',
        'fairness': {
            'sensitive_col':       'gender',
            'sensitive_condition': lambda x: x == 'Male',
            'target_positive':     '>50K',
            'label':               'Male vs Female',
        },
    },
    'bank_marketing': {
        'dataset_name':      'bank_marketing',
        'seeds':             [42, 43, 44],
        'target':            'deposit',
        'task':              'classification',
        'fairness': {
            'sensitive_col':       'age',
            'sensitive_condition': lambda x: x >= 30,
            'target_positive':     'yes',
            'label':               'Age >= 30 vs < 30',
        },
    },
    'california_housing': {
        'dataset_name': 'california_housing',
        'seeds':        [42, 43, 44],
        'target':       'median_house_value',
        'task':         'regression',
        'fairness':     None,
    },
    'cardiotocography': {
        'dataset_name': 'cardiotocography',
        'drop_cols': ['A', 'B', 'C', 'D', 'E', 'AD', 'DE', 'LD', 'FS', 'SUSP', 'NSP'],
        'seeds':        [42, 43, 44],
        'target':       'CLASS',
        'task':         'classification',
        'fairness':     None,
    },
    'compas': {
        'dataset_name': 'compas',
        'drop_cols': ['compas_screening_date', 'dob', 'in_custody', 'out_custody', 'c_charge_desc'],
        'seeds':        [42, 43, 44],
        'target':       'is_recid',
        'task':         'classification',
        'fairness': {
            'sensitive_col':       'race',
            'sensitive_condition': lambda x: x == 'African-American',
            'target_positive':     1,
            'label':               'African-American vs Other',
        },
    },
    'loan_prediction': {
        'dataset_name': 'loan_prediction',
        'seeds':        [42, 43, 44, 45, 46],
        'target':       'Loan_Status',
        'task':         'classification',
        'fairness': {
            'sensitive_col':       'Gender',
            'sensitive_condition': lambda x: x == 'Male',
            'target_positive':     'Y',
            'label':               'Male vs Female',
        },
    },
}

cfg          = CONFIGS[DATASET]
dataset_name = cfg['dataset_name']
seeds        = cfg['seeds']
target       = cfg['target']
task         = cfg['task']
fairness_cfg = cfg['fairness']

# Resolve framework aliases (e.g.: bank-marketing expands 'great' into two names)
aliases = cfg.get('framework_aliases', {})
llm_frameworks = []
for fw in LLM_FRAMEWORKS:
    llm_frameworks += aliases.get(fw, [fw])

methods = ['real'] + llm_frameworks + GANS

BASE_DIR      = f'../datasets/{DATASET}'
DATA_DIR      = f'{BASE_DIR}/data'
SYNTHETIC_DIR = f'{BASE_DIR}/synthetic-data'

print(f'Dataset          : {DATASET}')
print(f'Base Dir          : {BASE_DIR}')
print(f'Synthetic Dir          : {SYNTHETIC_DIR}')
print(f'Target           : {target}  ({task})')
print(f'Seeds            : {seeds}')
print(f'LLM frameworks   : {llm_frameworks}')
print(f'LLMs             : {LLMS}')
print(f'GANs             : {GANS}')
print(f'Methods          : {methods}')
if fairness_cfg:
    print(f'Fairness         : {fairness_cfg["label"]}')

## Load Data

In [ ]:
train_datasets = {}
test_datasets  = {}


# Real data
train_datasets['real'] = {}
for seed in seeds:
    train_datasets['real'][seed] = pd.read_csv(f'{DATA_DIR}/{dataset_name}_seed{seed}_train.csv')
    test_datasets[seed]          = pd.read_csv(f'{DATA_DIR}/{dataset_name}_seed{seed}_test.csv')

# LLM frameworks
for framework in llm_frameworks:
    train_datasets[framework] = {}
    for llm in LLMS:
        train_datasets[framework][llm] = {}
        for seed in seeds:
            path = f'{SYNTHETIC_DIR}/{framework}/{llm}/{dataset_name}_{framework}_{llm}_seed{seed}.csv'
            train_datasets[framework][llm][seed] = pd.read_csv(path)

# GANs
for gan in GANS:
    train_datasets[gan] = {}
    for seed in seeds:
        path = f'{SYNTHETIC_DIR}/{gan}/{dataset_name}_{gan}_seed{seed}.csv'
        train_datasets[gan][seed] = pd.read_csv(path)


if dataset_name == 'bank_marketing' and "tabfairgan" in GANS:
    for seed in seeds:
        train_datasets['tabfairgan'][seed].drop('age_group', axis=1, inplace=True)

print(f'Real train shape  : {train_datasets["real"][seeds[0]].shape}')
print(f'Real test shape   : {test_datasets[seeds[0]].shape}')
print(f'Columns ({len(train_datasets["real"][seeds[0]].columns)}): {list(train_datasets["real"][seeds[0]].columns)}')


## Cleaning of Invalid Categorical Values
Detects categorical values that do not exist in the real data (e.g.: "admin. primary" instead of "admin.") and tries to map them to the correct value. The corrections are applied in-place in `train_datasets`, so all the evaluations below see the already cleaned data.

In [ ]:
# Summary by (method, llm, seed) — before cleaning
invalid_summary = se.run_invalid_values_evaluation(train_datasets, methods, seeds)
display(invalid_summary)

In [ ]:
# Correction suggestions for the first synthetic method (1st seed)
_first_syn = next(m for m in methods if m != 'real')
_syn_sample = (
    train_datasets[_first_syn][LLMS[0]][seeds[0]]
    if _first_syn in llm_frameworks else train_datasets[_first_syn][seeds[0]]
)
_sugg = se.suggest_corrections(train_datasets['real'][seeds[0]], _syn_sample)
print(f'Suggestions for {_first_syn} / seed={seeds[0]}:')
_sugg

In [ ]:
# Apply corrections in-place on all synthetic datasets
fix_report = se.fix_invalid_values_in_train_datasets(train_datasets, methods, seeds)
print(f'Total cells corrected: {fix_report["n_corrected"].sum()} | '
      f'Total cells still invalid: {fix_report["n_unfixed"].sum()}')
display(fix_report)

## 1. Structural Analysis
Comparison of the structure, per-column statistics and distributions between the real and synthetic datasets.

In [ ]:
# 1.1 Structural summary (rows, columns, missings, duplicates)
structure_df = se.run_structure_summary(train_datasets, methods, seeds)
display(structure_df.round(4))

In [ ]:
# 1.2 Per-column statistics — real vs first synthetic method (1st seed)
first_syn = next(m for m in methods if m != 'real')
syn_sample = (
    train_datasets[first_syn][LLMS[0]][seeds[0]]
    if first_syn in llm_frameworks else train_datasets[first_syn][seeds[0]]
)
print(f'Comparison real vs {first_syn} (seed={seeds[0]})')
display(se.compare_column_summaries(train_datasets['real'][seeds[0]], syn_sample).round(4))

In [ ]:
# 1.3 Distances between distributions (KS, Wasserstein, TVD, JSD)
print('Running distribution distance evaluation...')
dist_df = se.run_distribution_distance_evaluation(train_datasets, methods, seeds)
display(dist_df.round(4))

In [ ]:
# 1.4 Category coverage (new / missing categories in the synthetic data)
print('Running categorical coverage evaluation...')
coverage_df = se.run_categorical_coverage_evaluation(train_datasets, methods, seeds)
display(coverage_df.round(4))

### 1.5 Visualizations

In [ ]:
# 1.5.1 Per-column distribution — real vs first synthetic method (1st seed)
fig = se.plot_column_distributions(
    train_datasets['real'][seeds[0]], syn_sample,
    title=f'real vs {first_syn} (seed={seeds[0]})',
)
plt.show()

In [ ]:
# 1.5.2 Correlation heatmaps (real / synthetic / |difference|)
fig = se.plot_correlation_heatmaps(
    train_datasets['real'][seeds[0]], syn_sample,
)
plt.show()

In [ ]:
# 1.5.3 Distances between distributions — mean per method (across seeds)
fig = se.plot_distance_summary(dist_df)
plt.show()

In [ ]:
# 1.5.4 Category coverage per method (mean across seeds)
fig = se.plot_categorical_coverage(coverage_df)
plt.show()

### 1.6 Distributions per dataset (real-train, real-test and each synthetic)

In [ ]:
# Collect {label: df} for the 1st seed: real_train, real_test and each synthetic method
datasets_by_seed = se.collect_datasets_by_seed(
    train_datasets, test_datasets, methods, seeds[0],
)
print('Datasets:', list(datasets_by_seed.keys()))

In [ ]:
# 1.6.1 Numeric columns — overlaid (step) histograms per dataset
fig = se.plot_numeric_distributions(
    datasets_by_seed,
    title=f'Numeric distributions (seed={seeds[0]})',
)
plt.show()

In [ ]:
# 1.6.2 Categorical columns — heatmap of relative frequencies (rows = dataset)
fig = se.plot_categorical_frequencies(
    datasets_by_seed,
    title=f'Categorical frequencies (seed={seeds[0]})',
)
plt.show()

In [ ]:
# 1.6.3 Categorical columns — grouped bar plot (one bar per dataset, top categories)
fig = se.plot_categorical_frequencies_bars(
    datasets_by_seed,
    title=f'Categorical frequencies — bar plot (seed={seeds[0]})',
)
plt.show()

## 2. Validity

In [ ]:
validity_df = se.run_validity_evaluation(train_datasets, methods, seeds)
display(validity_df)

## 3. Utility (TSTR)

In [ ]:
print('Running utility evaluation...')

if dataset_name == 'cardiotocography':
    drop_cols = ['A', 'B', 'C', 'D', 'E', 'AD', 'DE', 'LD', 'FS', 'SUSP', 'NSP']
else:
    drop_cols = None

utility_df = se.run_utility_evaluation(
    train_datasets, test_datasets, methods, seeds, target, task=task, drop_cols=drop_cols, dataset=dataset_name
)
display(utility_df.round(4))

## 4. Discriminator (Real vs Synthetic)

In [ ]:
print('Running discriminator evaluation...')
disc_df = se.run_discriminator_evaluation(train_datasets, methods, seeds)
display(disc_df.round(4))

## 5. Correlation Difference

In [ ]:
print('Running correlation evaluation...')
corr_df = se.run_correlation_evaluation(train_datasets, methods, seeds)
display(corr_df.round(4))

## 6. Similar Rows

In [ ]:
print('Running similar rows evaluation (may take a while)...')
dt = 0.95
# continuous_cols = ["fnlwgt", "CoapplicantIncome", "LoanAmount"]
thresholds = [1, 0.9, 0.8]
similar_df = se.run_percentage_similar_rows_evaluation(
    train_datasets, methods, seeds, test_datasets=test_datasets,
    thresholds=thresholds,
    # continuous_cols=continuous_cols
)
display(similar_df.round(2))

### 6.1 Per-column decomposition
For each synthetic row whose best match with real-train reaches the threshold, it checks in which columns a match occurred. `hit_rate` is the fraction of those rows in which the column matched; `baseline_rate` is the probability of matching by chance (under independent sampling of the marginals); `lift = hit_rate / baseline_rate`. Lift close to 1 indicates low-entropy columns ("matchable" by structure); high lift in high-cardinality columns indicates row-level memorization. The `real_test` baseline (computed on the real test data) tells you what to expect between two i.i.d. samples from the same distribution.

In [ ]:
# Per-column decomposition at a chosen threshold (e.g.: 0.9)
MATCH_THRESHOLD = 0.8
print(f'Running per-column decomposition at threshold t={MATCH_THRESHOLD}...')
match_decomp_df = se.run_match_column_contribution_evaluation(
    train_datasets, methods, seeds,
    threshold=MATCH_THRESHOLD,
    test_datasets=test_datasets,
)
display(match_decomp_df.round(4))

In [ ]:
# Heatmap of hit_rate (means across seeds)
fig = se.plot_match_column_contribution(
    match_decomp_df, value='hit_rate', annotate=True,
    title=f'hit_rate per column at t={MATCH_THRESHOLD} (mean across seeds)',
)
plt.show()

In [ ]:
# Heatmap of lift — columns with lift well above the real_test line suggest memorization
fig = se.plot_match_column_contribution(
    match_decomp_df, value='lift', annotate=True,
    title=f'lift per column at t={MATCH_THRESHOLD} (mean across seeds)',
)
plt.show()

### 6.2 Train-specific memorization vs marginal coverage
High lift against real_train can come from two sources: (i) row-level memorization (the LLM reproduces specific values seen during training) or (ii) marginal coverage (the LLM learned the right distribution, but does not copy rows).

To distinguish the two, we recompute the decomposition using real_test as the reference:
- `lift_train ≫ lift_test` → train-specific memorization.
- `lift_train ≈ lift_test` → marginal coverage (weaker signal).

In parallel, the set overlap `|syn ∩ real| / |syn|` on a high-cardinality column summarizes the same signal: if `overlap_train ≫ overlap_test`, the synthetic values come specifically from the training set.

In [ ]:
# 6.2.1 Per-column decomposition vs real_train and vs real_test (same threshold as 6.1)
print(f'Running decomposition vs train and vs test at t={MATCH_THRESHOLD}...')
decomp_tr_te = se.run_match_column_contribution_train_vs_test(
    train_datasets, test_datasets, methods, seeds,
    threshold=MATCH_THRESHOLD,
)

# Pivot and sort by gap (lift_train - lift_test); +inf collapses to NaN for visualization.
lift_pivot = (decomp_tr_te
              .replace([np.inf, -np.inf], np.nan)
              .pivot_table(index=['method', 'llm', 'column', 'dtype'],
                           columns='reference', values='lift', aggfunc='mean')
              .reset_index()
              .rename(columns={'train': 'lift_train', 'test': 'lift_test'}))
lift_pivot['lift_gap'] = lift_pivot['lift_train'] - lift_pivot['lift_test']
lift_pivot = lift_pivot.sort_values('lift_gap', ascending=False)
lift_pivot = lift_pivot[['method', 'llm', 'column', 'dtype', 'lift_train', 'lift_test', 'lift_gap']]
print('Top 15 (method, column) by lift_gap (train-specific memorization):')
display(lift_pivot.round(2))

In [ ]:
# 6.2.2 Value overlap: |syn ∩ real_train| / |syn| vs |syn ∩ real_test| / |syn|
# Column chosen automatically: the one with the largest lift_gap (numeric preferred).
_num_top = lift_pivot[lift_pivot['dtype'] == 'numeric'].dropna(subset=['lift_train'])
OVERLAP_COLUMN = _num_top.iloc[0]['column'] if not _num_top.empty else lift_pivot.iloc[0]['column']
print(f'Chosen column: {OVERLAP_COLUMN!r}')

overlap_df = se.run_value_overlap_evaluation(
    train_datasets, methods, seeds,
    test_datasets=test_datasets,
    columns=[OVERLAP_COLUMN],
)

# Show overlap_train and overlap_test side by side (means across seeds).
overlap_pivot = (overlap_df
                 .pivot_table(index=['method', 'llm'], columns='reference',
                              values='overlap', aggfunc='mean')
                 .reset_index()
                 .rename(columns={'train': 'overlap_train', 'test': 'overlap_test'}))
overlap_pivot['overlap_gap'] = overlap_pivot['overlap_train'] - overlap_pivot['overlap_test']
overlap_pivot = overlap_pivot.sort_values('overlap_gap', ascending=False)
print(f'Overlap on "{OVERLAP_COLUMN}" (mean across seeds):')
display(overlap_pivot.round(3))

### 6.3 Similarity curve vs threshold (train vs test as reference)
For each synthetic method, we compute the percentage of synthetic rows that reach each threshold *t* against real_train (solid line) and against real_test (dashed line). We overlay the real_test → real_train curve (black dotted line) as an i.i.d. baseline: two independent samples from the same distribution.

Reading:
- syn-vs-train **above** the dotted baseline → memorization (the LLM is closer to real_train than a fresh real sample from the same distribution).
- syn-vs-train **overlapping** the baseline → marginal coverage without memorization.
- syn-vs-train **below** the baseline → the generator underfits the train (typical of GANs).

The gap between syn-vs-train and syn-vs-test, on its own, overestimates memorization because real_train is typically larger than real_test and the generator was fit to real_train. The i.i.d. baseline absorbs these two confounders.

In [ ]:
# Can be heavy; with few methods and seeds it takes ~minutes per pair.
print('Running threshold curve syn-vs-train and syn-vs-test...')
tt_thresholds = [1.0, 0.95, 0.9, 0.85, 0.8, 0.75, 0.7]
# tt_thresholds = [1.0, 0.9, 0.8, 0.7]
curve_df = se.run_percentage_similar_rows_train_vs_test(
    train_datasets, test_datasets, methods, seeds,
    thresholds=tt_thresholds,
)
display(curve_df.round(2))

In [ ]:
# Plot per (method, llm) — solid curve = vs train, dashed = vs test
fig = se.plot_similarity_threshold_curves(
    curve_df,
    title=f'Similarity ≥ t — syn vs train (solid) and syn vs test (dashed)',
)
plt.show()

In [ ]:
b = (curve_df[curve_df['method'] == 'real_test']
   .groupby('seed')[[c for c in curve_df.columns if c.startswith('pct_')]]
   .mean())  # baseline per seed

syn = curve_df[curve_df['reference'] == 'train'].copy()
syn = syn[syn['method'] != 'real_test']

pct_cols = [c for c in curve_df.columns if c.startswith('pct_')]
for c in pct_cols:
  syn[f'premium_{c}'] = syn.apply(lambda r: r[c] - b.loc[r['seed'], c], axis=1)

premium = (syn
         .groupby(['method', 'llm'])[[f'premium_{c}' for c in pct_cols]]
         .mean()
         .round(2))
display(premium)

## 7. Distance to Closest Record (DCR)

In [ ]:
print('Running DCR evaluation (may take a while)...')
dcr_df = se.run_dcr_evaluation(train_datasets, methods, seeds)
display(dcr_df.round(4))

## 8. Membership Inference Attack (MIA)

In [ ]:
print('Running MIA evaluation (may take a while)...')
mia_df = se.run_mia_evaluation(train_datasets, test_datasets, methods, seeds)
display(mia_df.round(4))

## 9. Fairness (Discrimination Score)

In [ ]:
if fairness_cfg is None:
    print(f'Fairness not configured for dataset "{DATASET}".')
else:
    print(f'Running fairness evaluation ({fairness_cfg["label"]})...')
    fairness_df = se.run_fairness_evaluation(
        train_datasets, methods, seeds,
        sensitive_col       = fairness_cfg['sensitive_col'],
        sensitive_condition = fairness_cfg['sensitive_condition'],
        target              = target,
        target_positive     = fairness_cfg['target_positive'],
    )
    display(fairness_df.round(4))

## 10. Overall Summary

In [ ]:
metric_cols = ['f1', 'auc', 'acc'] if task == 'classification' else ['r2', 'rmse', 'mae']

frames = [
    validity_df.set_index(['method', 'llm', 'seed'])[['validity_rate']],
    utility_df.set_index(['method', 'llm', 'seed'])[metric_cols],
    disc_df.set_index(['method', 'llm', 'seed'])[['disc_auc']],
    corr_df.set_index(['method', 'llm', 'seed'])[['corr_mae']],
    dcr_df.set_index(['method', 'llm', 'seed'])[['dcr_median']],
    mia_df.set_index(['method', 'llm', 'seed'])[['mia_auc']],
    similar_df.set_index(['method', 'llm', 'seed'])[[c for c in similar_df.columns if c.startswith('pct_')]],
]

if fairness_cfg is not None:
    frames.append(fairness_df.set_index(['method', 'llm', 'seed'])[['ds', 'p_y1_s1', 'p_y1_s0']])

summary_all = pd.concat(frames, axis=1).reset_index()
print(f'=== Overall Summary — {DATASET} ===')
display(summary_all.round(4))

In [ ]:
import os

# ── Save metrics to Excel ─────────────────────────────────────────────────
# Rows: (method, llm)  |  Columns: metric_seedXX
results_dir = f'{BASE_DIR}/results'
os.makedirs(results_dir, exist_ok=True)

excel_path = f'{results_dir}/{dataset_name}_metrics.xlsx'

# Metric columns (everything except method, llm, seed)
metric_cols = [c for c in summary_all.columns if c not in ('method', 'llm', 'seed')]

# Pivot: rows = (method, llm), columns = metric_seedXX
pivot = summary_all.set_index(['method', 'llm', 'seed'])[metric_cols].unstack('seed')
pivot.columns = [f'{metric}_seed{seed}' for metric, seed in pivot.columns]

# Reorder columns: group by metric (f1_42, f1_43, f1_44, auc_42, ...)
ordered_cols = [f'{m}_seed{s}' for m in metric_cols for s in seeds]
ordered_cols = [c for c in ordered_cols if c in pivot.columns]
excel_df = pivot[ordered_cols].reset_index()

# Sort rows by method and llm in the defined order
method_order = ['real', 'real_test', 'great', 'tabula', 'taptap'] + GANS
# method_order = ['real', 'real_test'] + GANS
llm_order    = ['distilgpt2', 'taptap_distill', 'qwen3_03B_distil', '-']
method_cat = pd.Categorical(excel_df['method'], categories=method_order, ordered=True)
llm_cat    = pd.Categorical(excel_df['llm'],    categories=llm_order,    ordered=True)
excel_df = excel_df.assign(method=method_cat, llm=llm_cat).sort_values(['method', 'llm']).reset_index(drop=True)

excel_df.to_excel(excel_path, index=False)
print(f'Metrics saved to: {excel_path}')
print(f'Shape: {excel_df.shape}')
display(excel_df)